# RetailPulse — AI-Powered Customer Analytics & Demand Forecasting
### One-day build: EDA → RFM Segmentation → Demand Forecast → Churn Prediction → Inventory Optimization → Dashboard

Run cells top to bottom. Everything after the data-load cell works on the
real **UCI Online Retail** dataset with zero changes.


## 0. Setup — install libraries

In [ ]:
%pip install -q prophet xgboost shap pandas numpy scikit-learn matplotlib seaborn openpyxl statsmodels streamlit pyngrok


: 

## 1. Load the dataset (Day 1)

The UCI **Online Retail** dataset is an Excel file. We load it directly
from the UCI URL — no manual upload needed. If the UCI server is slow/down,
fall back to uploading the file manually (`files.upload()` cell is commented
below).

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = r"C:\Users\ASUS\OneDrive\Desktop\AI-Powered-Demand-Inventory-Intelligence\Online_Retail_UTF8.csv.xlsx"

try:
    df = pd.read_excel(DATA_URL)
    print("Loaded from UCI URL:", df.shape)
except Exception as e:
    print("Direct URL load failed:", e)
    print("Falling back to manual upload. Run the next cell.")


In [ ]:
# --- OPTIONAL fallback: only run this if the URL cell above failed ---
# from google.colab import files
# uploaded = files.upload()  # choose Online Retail.xlsx from your computer
# df = pd.read_excel(list(uploaded.keys())[0])
# print(df.shape)


In [ ]:
if 'df' in locals():
    df.head()
else:
    print("Error: df not loaded. Check cell 4 for errors, or run the fallback upload in cell 5.")

## 2. Initial EDA (Day 1)
Distribution analysis, missing values, correlation heatmap — before any cleaning,
so we can see exactly what's messy in the raw extract.

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum())
print("\nDtypes:\n", df.dtypes)
df.describe(include="all").T


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df["Quantity"].clip(-50, 50).hist(bins=50, ax=axes[0]); axes[0].set_title("Quantity (clipped)")
df["UnitPrice"].clip(0, 50).hist(bins=50, ax=axes[1]); axes[1].set_title("UnitPrice (clipped)")
df["Country"].value_counts().head(10).plot(kind="bar", ax=axes[2]); axes[2].set_title("Top 10 Countries")
plt.tight_layout(); plt.show()


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(5,4))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (raw numeric columns)")
plt.show()


## 3. Data cleaning + feature engineering (Day 2)
Standard cleaning for this dataset: drop rows with no CustomerID (can't
attribute demand or segment a customer we can't identify), drop
negative/zero Quantity or UnitPrice (returns and data errors, out of scope
for a forward demand model), dedupe, and build `TotalPrice`.


In [ ]:
n0 = len(df)
df = df.dropna(subset=["CustomerID"]).copy()
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df.drop_duplicates()
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

print(f"Cleaned: {n0:,} -> {len(df):,} rows ({(1 - len(df)/n0):.1%} removed)")
df.head()


In [ ]:
# Great-Expectations-style validation, kept lightweight (full GE setup is
# overkill for a one-day build) -- these are the checks that matter:
assert df["Quantity"].min() > 0, "Negative/zero quantity slipped through"
assert df["UnitPrice"].min() > 0, "Negative/zero price slipped through"
assert df["CustomerID"].isna().sum() == 0, "Null CustomerID slipped through"
assert df.duplicated().sum() == 0, "Duplicates slipped through"
print("All validation checks passed.")


## 4. Customer Segmentation — RFM + KMeans (Day 3)
Recency, Frequency, Monetary per customer, scaled, then KMeans. We check
silhouette score across a range of k rather than hard-coding 4 clusters,
so the cluster count is chosen by the data, not by assumption.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

scores = {}
for k in range(3, 9):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(rfm_scaled)
    scores[k] = silhouette_score(rfm_scaled, labels)
    print(f"k={k}: silhouette={scores[k]:.3f}")

best_k = max(scores, key=scores.get)
print(f"\nChosen k = {best_k}")
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean().round(1)


In [ ]:
# Business interpretation: label clusters by their RFM profile rather than
# just calling them "Cluster 0/1/2..."
cluster_profile = rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean()

def label_cluster(row, medians):
    if row["Monetary"] >= medians["Monetary"] and row["Recency"] <= medians["Recency"]:
        return "Champions"
    if row["Recency"] > medians["Recency"] * 1.5:
        return "At Risk / Churned"
    if row["Frequency"] >= medians["Frequency"]:
        return "Loyal"
    return "Occasional"

medians = rfm[["Recency","Frequency","Monetary"]].median()
cluster_profile["Segment"] = cluster_profile.apply(lambda r: label_cluster(r, medians), axis=1)
print(cluster_profile)

rfm["Segment"] = rfm["Cluster"].map(cluster_profile["Segment"])


In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=rfm, x="Recency", y="Monetary", hue="Segment", palette="Set2")
plt.title("Customer Segments — Recency vs Monetary")
plt.show()


## 5. Time-series prep + stationarity check (Day 4)
Aggregate to daily revenue, check stationarity with an ADF test, and look
at the seasonal decomposition before fitting anything.


In [ ]:
daily = df.groupby(df["InvoiceDate"].dt.date)["TotalPrice"].sum().reset_index()
daily.columns = ["ds", "y"]
daily["ds"] = pd.to_datetime(daily["ds"])
daily = daily.set_index("ds").asfreq("D").fillna(0).reset_index()

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

adf_result = adfuller(daily["y"])
print(f"ADF statistic: {adf_result[0]:.3f}, p-value: {adf_result[1]:.4f}")
print("-> Series is", "stationary" if adf_result[1] < 0.05 else "NOT stationary (has trend/seasonality Prophet needs to model)")

decomposition = seasonal_decompose(daily.set_index("ds")["y"], period=7, model="additive")
fig = decomposition.plot()
fig.set_size_inches(10, 6)
plt.tight_layout(); plt.show()


## 6. Demand Forecasting — Prophet (Day 5)
Prophet, not an LSTM ensemble — for a one-day build, Prophet gets you a
solid, interpretable forecast with uncertainty intervals fast. An LSTM
ensemble is a stretch goal, not core scope, once this baseline is working.


In [ ]:
from prophet import Prophet

m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
m.fit(daily)

future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)

fig1 = m.plot(forecast)
plt.title("30-Day Revenue Forecast")
plt.show()

fig2 = m.plot_components(forecast)
plt.show()

forecast[["ds","yhat","yhat_lower","yhat_upper"]].tail(10)


In [ ]:
# Simple backtest: hold out the last 30 days, fit on the rest, compare WAPE
train = daily.iloc[:-30]
test = daily.iloc[-30:]

m_bt = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
m_bt.fit(train)
future_bt = m_bt.make_future_dataframe(periods=30)
forecast_bt = m_bt.predict(future_bt)
pred = forecast_bt.set_index("ds").loc[test["ds"], "yhat"].values

def wape(y_true, y_pred):
    return np.abs(y_true - y_pred).sum() / np.abs(y_true).sum()

# naive baseline: predict last known value repeated
naive_pred = np.repeat(train["y"].tail(7).mean(), len(test))

print(f"WAPE (Prophet):        {wape(test['y'].values, pred):.3f}")
print(f"WAPE (naive baseline): {wape(test['y'].values, naive_pred):.3f}")


## 7. Churn Prediction — XGBoost (Day 9)
Define churn using a cutoff **before** the end of the data (not "did they
buy in the last N days of the whole dataset" — that would only tell you
who bought recently, trivially). Features are built only from the training
window, label from what happens after — no leakage.


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

cutoff = df["InvoiceDate"].min() + (df["InvoiceDate"].max() - df["InvoiceDate"].min()) * 0.6
train_window = df[df["InvoiceDate"] <= cutoff]
label_window = df[df["InvoiceDate"] > cutoff]

feat = train_window.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (cutoff - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
    AvgBasket=("TotalPrice", "mean"),
)
active_after = set(label_window["CustomerID"].unique())
feat["Churned"] = (~feat.index.isin(active_after)).astype(int)
print("Churn rate in training window:", feat["Churned"].mean().round(3))

X, y = feat[["Recency","Frequency","Monetary","AvgBasket"]], feat["Churned"]
# Guard against too few members in the minority class for stratified split
can_stratify = y.value_counts().min() >= 2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y if can_stratify else None
)
if not can_stratify:
    print("Warning: too few churned customers to stratify -- using a plain random split. "
          "Consider an earlier cutoff (more history counted as 'after') if this happens on real data.")

clf = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, eval_metric="logloss")
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]

print("AUC-ROC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, (proba > 0.5).astype(int)))


In [ ]:
import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, show=True)


## 8. Inventory Optimization Recommendation (Day 10)
Translate the 30-day revenue forecast into a stock recommendation. This is
a simplified, transparent rule (not a solver) -- appropriate for a one-day
build and easy to explain to a non-technical stakeholder.


In [ ]:
next_30 = forecast[["ds","yhat"]].tail(30)
recommended_revenue = next_30["yhat"].sum()
print(f"Recommended stock value for next 30 days: Rs {recommended_revenue:,.0f}")

# Per-top-product recommendation (simple share-of-revenue split)
top_products = df.groupby("Description")["TotalPrice"].sum().sort_values(ascending=False).head(10)
product_share = top_products / top_products.sum()
recommended_by_product = (product_share * recommended_revenue).round(0)
recommended_by_product.to_frame("Recommended_Stock_Value")


## 9. Package as a Streamlit Dashboard (Day 15-18, compressed)
Colab can't natively serve Streamlit, so we write the app to a file and
tunnel it with `pyngrok` for a live preview. For your real submission,
deploy this same file to **Streamlit Community Cloud** (free, public URL,
no tunnel needed) — see the note at the bottom of this cell.


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="RetailPulse", layout="wide")
st.title("📊 RetailPulse — Customer Analytics & Demand Forecasting")

st.markdown("""
This dashboard is generated from the same cleaned data and models built in
the companion Colab notebook. Re-run the notebook's export cell to refresh
`rfm.csv` / `forecast.csv` / `churn_scores.csv` before restarting this app.
""")

@st.cache_data
def load():
    rfm = pd.read_csv("rfm.csv")
    forecast = pd.read_csv("forecast.csv", parse_dates=["ds"])
    return rfm, forecast

try:
    rfm, forecast = load()
except FileNotFoundError:
    st.warning("Run the notebook's export cell first to create rfm.csv and forecast.csv.")
    st.stop()

tab1, tab2, tab3 = st.tabs(["Customer Segments", "Demand Forecast", "Reorder Guidance"])

with tab1:
    st.subheader("Customer Segments (RFM + KMeans)")
    st.dataframe(rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].mean().round(1))
    st.bar_chart(rfm["Segment"].value_counts())

with tab2:
    st.subheader("30-Day Revenue Forecast")
    st.line_chart(forecast.set_index("ds")[["yhat","yhat_lower","yhat_upper"]].tail(60))

with tab3:
    st.subheader("Recommended Stock Value (next 30 days)")
    total = forecast["yhat"].tail(30).sum()
    st.metric("Total recommended stock value", f"Rs {total:,.0f}")


In [ ]:
# Export the data the dashboard needs
rfm.reset_index().to_csv("rfm.csv", index=False)
forecast.to_csv("forecast.csv", index=False)
print("Exported rfm.csv and forecast.csv for the dashboard.")


In [ ]:
# --- Run the dashboard from Colab via a tunnel (for a quick live check only) ---
# 1. Get a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# 2. Uncomment and run:
#
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_TOKEN_HERE")
# !streamlit run app.py &>/content/logs.txt &
# public_url = ngrok.connect(8501)
# print(public_url)
#
# For the actual submission, push app.py + requirements.txt to GitHub and
# deploy on https://share.streamlit.io instead -- it's free, public, and
# doesn't depend on Colab staying open.


## 10. Summary (for your README / submission)
Run this last cell to print the key numbers you'll quote in your
documentation and demo video.


In [ ]:
print("=== RetailPulse — Key Results ===")
print(f"Customers segmented: {len(rfm):,} into {rfm['Segment'].nunique()} segments")
print(f"Forecast WAPE (Prophet) vs naive baseline: see Section 6 backtest cell above")
print(f"Churn model AUC-ROC: see Section 7 output above")
print(f"Recommended 30-day stock value: Rs {forecast['yhat'].tail(30).sum():,.0f}")
